# Automated Underwriting Decision Tree

### Retail Loan Mortgage Approval — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of retail lending and mortgage approval terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the retail lending and mortgage approval problem and why the chosen classification approach suits this banking workflow.
- Implement an end-to-end classification pipeline on synthetic data and evaluate performance with domain-appropriate metrics.
- Assess whether the model is operationally viable, considering error costs, fairness, and the limitations of synthetic data.

**Estimated time:** 35–45 minutes

**Collection context:** Mortgage risk, automated underwriting, appraisal valuation, and loan approval processes in retail banking.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** In banking, you can't just say "the computer said no." You have to explain *why*. Decision Trees are 'white-box' models, meaning we can see exactly what rules they use (e.g., "If Credit Score > 650 AND Income > $50k, then Approve"). This makes them perfect for automated underwriting.

**Input data used:** Credit Score, Annual Income, Employment Length, Loan Amount.

**What we predict:** Loan Status (`Approved` vs `Rejected`).

**ML method used:** Decision Tree Classifier.

**Learning goal:** Understand how machine learning can turn complex data into simple, executable business rules.

**Primary stakeholders:** mortgage officers, underwriters, credit risk managers, and compliance teams.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Underwriting Data

We generate 1,000 applications with a clear underlying rule set.

In [ ]:
n = 1000
credit_score = RNG.integers(500, 850, n)
income = RNG.integers(20000, 150000, n)
employment_yrs = RNG.integers(0, 20, n)

# Simple Rule for Approval:
# (Credit Score > 680) OR (Credit Score > 620 AND Income > 60000 AND Employment > 2)
approved = (
    (credit_score > 680) | 
    ((credit_score > 620) & (income > 60000) & (employment_yrs > 2))
).astype(int)

df = pd.DataFrame({
    'credit_score': credit_score,
    'income': income,
    'employment_yrs': employment_yrs,
    'approved': approved
})

print(df.head())

## Step 4 — Exploratory Data Analysis

For this workflow, primary exploration happens through the operational visualisations in later steps.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

In [ ]:
# Features are used as created in Step 3.
print("Target column: 'approved'")

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the majority class ---
baseline_clf = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
baseline_clf.fit(X_train, y_train)
baseline_pred = baseline_clf.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_pred)
print(f"Baseline accuracy (majority-class): {baseline_acc:.3f}")
print(f"Any useful model must beat this {baseline_acc:.3f} floor.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

We'll limit the 'depth' of the tree to keep the rules simple.

In [ ]:
X = df.drop('approved', axis=1)
y = df['approved']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

clf = DecisionTreeClassifier(max_depth=3)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred):.1%}")

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(
    clf, 
    feature_names=X.columns, 
    class_names=['Reject', 'Approve'], 
    filled=True, 
    rounded=True, 
    fontsize=12
)
plt.title("Automated Underwriting Logic (Decision Tree)")
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Visualisation titled "Automated Underwriting Logic (Decision Tree)". The chart highlights key patterns that inform the modelling approach.

In [ ]:
app_1 = [[640, 75000, 5]] # Good income/exp, fair credit
app_2 = [[600, 30000, 1]] # Low credit, low income

def get_decision(app):
    pred = clf.predict(app)[0]
    return "Approved" if pred == 1 else "Rejected"

print(f"Applicant 1 Decision: {get_decision(app_1)}")
print(f"Applicant 2 Decision: {get_decision(app_2)}")

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

1. **Transparency:** By looking at the tree, we can see that 'credit_score' was the most important first split.
2. **Consistency:** Unlike humans, the model will always give the same decision for the same data, eliminating bias (as long as the training data is fair).
3. **Regulation:** Banks use these models to prove to regulators that their lending criteria are objective and follow the law.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: score representative cases from the test set ---
print("Operational demonstration — model decision support:\n")
proba = clf.predict_proba(X_test)[:, 1]
low_idx  = int(np.argmin(proba))
high_idx = int(np.argmax(proba))
mid_idx  = int(np.argsort(proba)[len(proba) // 2])

for label, idx in [("Low-risk", low_idx), ("Medium-risk", mid_idx), ("High-risk", high_idx)]:
    row = X_test.iloc[idx]
    prob = proba[idx]
    print(f"{label} case  predicted probability: {prob:.1%}")
    print(f"  Features: {dict(row.round(2))}")
    print()

print("A decision-maker would combine these scores with business rules and domain judgement.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end retail lending and mortgage approval workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Mortgage models must comply with fair lending laws and avoid discriminatory approval patterns.
- Automated underwriting can disadvantage applicants from historically underserved communities.
- Transparency in loan decisions is a regulatory requirement, not an optional feature.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in retail lending and mortgage approval?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.